# Bi-Objective Optimization: Four Solution Approaches

## 0. Setup

The notebook uses two optimization packages:

- **GAMSpy** (`gp`) builds and solves the mathematical-programming models
- **pymoo** supplies the NSGA-II genetic algorithm

Run the setup cells in order. The reference data downloads automatically from the public tutorial repository; no Drive mounting or manual upload is needed. Section 2.4 checks the cached objective scaling and rebuilds the reference fronts only if needed (or if `REBUILD_REFERENCE = True`). Rebuilding can take a long time.

Save a copy in Drive to keep your work, then complete each **To Do** before running its method.


In [ ]:
# Run this once to install the two optimization packages.
%pip install -q gamspy pymoo

In [ ]:
# General Python tools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy.stats import qmc

# GAMSPY: mathematical-programming models
import gamspy as gp

# PYMOO: NSGA-II genetic algorithm
from pymoo.core.problem import ElementwiseProblem
from pymoo.core.variable import Real, Integer
from pymoo.core.mixed import (MixedVariableSampling, MixedVariableMating,
                              MixedVariableDuplicateElimination)
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

# Fix random numbers so repeated runs give comparable results.
np.random.seed(0)

## 1. Three Problems

$$
\textbf{MOP1:} \quad
\min f_1(\mathbf{x}) = \frac{1}{x_1^2+x_2^2+1}, \qquad
\min f_2(\mathbf{x}) = x_1^2+3x_2^2+4, \qquad
-3 \le x_1,x_2 \le 3
$$

$$
\textbf{SCH2:} \quad
\min f_1(\mathbf{x}) = (x_1+x_2-7.5)^2+\frac{(x_2-x_1+3)^2}{4}, \qquad
\min f_2(\mathbf{x}) = \frac{(x_1-1)^2}{4}+\frac{(x_2-4)^2}{2}
$$
$$
\text{s.t.} \quad \frac{(x_1-2)^3}{2}+x_2-2.5 \le 0, \qquad
x_1+x_2-8(x_2-x_1+0.65)^2-3.85 \le 0, \qquad
0\le x_1\le5,\ 0\le x_2\le3
$$

### Test 16: mixed-integer benchmark

Test 16 is taken from Eichfelder et al. [1].

$$
\begin{aligned}
\min \quad & f_1(\mathbf{x})=x_1+x_3, \\
\min \quad & f_2(\mathbf{x})=x_2+x_4.
\end{aligned}
$$

The feasible region is

$$
x_1^2+x_2^2\ge1,\qquad x_3^2+x_4^2\le9,
$$
$$
(x_1,x_2)\in[0,1]^2,\qquad
(x_3,x_4)\in\{-3,-2,-1,0,1,2,3\}^2.
$$

In [ ]:
# ======================== GAMSPY MODEL DEFINITIONS ========================
# Each function receives a GAMSpy container and adds one benchmark model.

def build_mop1(m):
    # 1. Create decision variables.
    x1 = gp.Variable(m, "x1", type="free")
    x2 = gp.Variable(m, "x2", type="free")
    x1.lo[...] = -3
    x1.up[...] = 3
    x2.lo[...] = -3
    x2.up[...] = 3

    # 2. Create variables that hold the two objective values.
    f1 = gp.Variable(m, "f1", type="free")
    f2 = gp.Variable(m, "f2", type="free")
    eq1 = gp.Equation(m, "eq_f1")
    eq1[...] = f1 == 1 / (x1**2 + x2**2 + 1)
    eq2 = gp.Equation(m, "eq_f2")
    eq2[...] = f2 == x1**2 + 3 * x2**2 + 4

    # Return the pieces needed by the solution methods below.
    return dict(x=[x1, x2], f1=f1, f2=f2)


def build_sch2(m):
    # Decision variables and their bounds
    x1 = gp.Variable(m, "x1", type="free")
    x2 = gp.Variable(m, "x2", type="free")
    x1.lo[...] = 0
    x1.up[...] = 5
    x2.lo[...] = 0
    x2.up[...] = 3

    f1 = gp.Variable(m, "f1", type="free")
    f2 = gp.Variable(m, "f2", type="free")
    eq1 = gp.Equation(m, "eq_f1")
    eq1[...] = f1 == (x1 + x2 - 7.5) ** 2 + (x2 - x1 + 3) ** 2 / 4
    eq2 = gp.Equation(m, "eq_f2")
    eq2[...] = f2 == (x1 - 1) ** 2 / 4 + (x2 - 4) ** 2 / 2

    con1 = gp.Equation(m, "con1")
    con1[...] = (x1 - 2) ** 3 / 2 + x2 - 2.5 <= 0
    con2 = gp.Equation(m, "con2")
    con2[...] = x1 + x2 - 8 * (x2 - x1 + 0.65) ** 2 - 3.85 <= 0
    return dict(x=[x1, x2], f1=f1, f2=f2)


def build_test16(m):
    # x1 and x2 are continuous variables.
    x1 = gp.Variable(m, "x1", type="free")
    x1.lo[...] = 0
    x1.up[...] = 1
    x2 = gp.Variable(m, "x2", type="free")
    x2.lo[...] = 0
    x2.up[...] = 1

    # x3 and x4 must take integer values.
    x3 = gp.Variable(m, "x3", type="integer")
    x3.lo[...] = -3
    x3.up[...] = 3
    x4 = gp.Variable(m, "x4", type="integer")
    x4.lo[...] = -3
    x4.up[...] = 3

    f1 = gp.Variable(m, "f1", type="free")
    f2 = gp.Variable(m, "f2", type="free")
    eq1 = gp.Equation(m, "eq_f1")
    eq1[...] = f1 == x1 + x3
    eq2 = gp.Equation(m, "eq_f2")
    eq2[...] = f2 == x2 + x4
    outside_ball = gp.Equation(m, "outside_ball")
    outside_ball[...] = x1**2 + x2**2 >= 1
    integer_ball = gp.Equation(m, "integer_ball")
    integer_ball[...] = x3**2 + x4**2 <= 9
    return dict(x=[x1, x2, x3, x4], f1=f1, f2=f2)


# Match each problem name to its builder and GAMSpy problem type.
BUILDERS = {"MOP1": build_mop1, "SCH2": build_sch2, "Test 16": build_test16}
PROBLEM_TYPES = {"MOP1": "NLP", "SCH2": "NLP", "Test 16": "MINLP"}

# Stores the normalised (f1, f2) points produced by every method.
results = {"Weighted sum": {}, "Epsilon-constraint": {}, "Modified NBI": {}, "NSGA-II": {}}

# For the scalarization methods we also keep the input parameter (w1, epsilon, beta1)
# that produced each point, so the plots can colour the points by that parameter.
method_params = {method: {} for method in results}


def solve_succeeded(model):
    # GAMSpy uses Integer for a feasible mixed-integer incumbent.
    return any(label in str(model.status) for label in ("Optimal", "Integer"))

## 2. Anchor points, the payoff matrix and the objective ranges (GAMSpy)

**Anchor points** are the individual minimizers of $f_1$ alone and $f_2$ alone.
Their objective vectors form the columns of the **payoff matrix** $\Phi$. With two
objectives, this single matrix contains both corners of the front:

$$
\Phi = \begin{bmatrix} f_1(\mathbf{x}^{*1}) & f_1(\mathbf{x}^{*2}) \\
f_2(\mathbf{x}^{*1}) & f_2(\mathbf{x}^{*2}) \end{bmatrix},
\qquad
f^{id} = \mathrm{diag}(\Phi),
\qquad
f^{nad} = (\Phi_{12},\ \Phi_{21})
$$

the **ideal point** $f^{id}$ on the diagonal (the best value each objective can reach)
and the **nadir point** $f^{nad}$ on the off-diagonal (the value each objective takes at
the *other* anchor, that is, its worst value along the front). The payoff matrix is computed once per problem.

**Multi-start:** each anchor minimizes one objective with GAMSpy's NLP/MINLP solver.
On a non-convex problem (SCH2 here), a single solve from one starting point is not
reliable, since it can converge to a *local* optimum instead of the true global minimum
(concretely: SCH2's $f_1$ anchor from an unseeded solve lands at $f_1\approx8.07$,
but the true global minimum is $f_1\approx7.25$). Since every method below
normalizes against $f^{id}$/$f^{nad}$, a wrong anchor silently distorts every
scalarization. `get_anchor_points` therefore evaluates each objective from
`N_ANCHOR_STARTS` **Sobol-sequence** starting points spread over the decision
variables' own bounds, a low-discrepancy design that covers the box more evenly
than uniform random points for the same sample size, plus the solver's own
unseeded default, and keeps whichever converges to the best value.

In [ ]:
anchor_x = {}

N_ANCHOR_STARTS = 16  # 2**4: a power of 2 gives a properly balanced Sobol sequence
ANCHOR_SEED = 0


def _var_bound_info(name):
    # Read each decision variable's own declared bounds/type from a throwaway
    # build, so Sobol starting points stay valid without hardcoding bounds here.
    m = gp.Container()
    comp = BUILDERS[name](m)
    return [(v.name, float(v.lo.records["lower"].values[0]),
             float(v.up.records["upper"].values[0]), v.type)
            for v in comp["x"]]


def _sobol_starts(name, n_starts, seed):
    info = _var_bound_info(name)
    lo = np.array([b[1] for b in info])
    hi = np.array([b[2] for b in info])
    sampler = qmc.Sobol(d=len(info), scramble=True, seed=seed)
    sample = qmc.scale(sampler.random(n=n_starts), lo, hi)
    starts = []
    for row in sample:
        point = {}
        for (vname, _, _, vtype), val in zip(info, row):
            point[vname] = float(round(val)) if vtype == "integer" else float(val)
        starts.append(point)
    return starts


def _solve_single_objective(name, obj_key, start):
    m = gp.Container()
    comp = BUILDERS[name](m)
    if start is not None:
        for v in comp["x"]:
            if v.name in start:
                v.l[...] = start[v.name]  # initial level = starting guess, not a bound
    model = gp.Model(m, name=f"{name.replace(' ', '_').lower()}_{obj_key}_try",
                      equations=m.getEquations(), problem=PROBLEM_TYPES[name],
                      sense="MIN", objective=comp[obj_key])
    model.solve()
    if solve_succeeded(model):
        f1v = comp["f1"].records.level.values[0]
        f2v = comp["f2"].records.level.values[0]
        xsol = {v.name: float(v.records.level.values[0]) for v in comp["x"]}
        return f1v, f2v, xsol
    return None


def get_anchor_points(name, n_starts=N_ANCHOR_STARTS, seed=ANCHOR_SEED):
    starts = [None] + _sobol_starts(name, n_starts, seed)  # None = solver's own default
    Phi = np.zeros((2, 2))
    x_at_anchor = {}
    for j, obj_key in enumerate(["f1", "f2"]):
        best = None
        for start in starts:
            result = _solve_single_objective(name, obj_key, start)
            if result is None:
                continue
            f1v, f2v, xsol = result
            val = f1v if obj_key == "f1" else f2v
            if best is None or val < best[0]:
                best = (val, f1v, f2v, xsol)
        _, f1v, f2v, xsol = best
        Phi[0, j] = f1v
        Phi[1, j] = f2v
        # Remember the decision vector that produced this anchor.
        x_at_anchor[obj_key] = xsol
    anchor_x[name] = x_at_anchor
    f_id = np.diag(Phi).copy()
    return Phi, f_id


anchors = {name: get_anchor_points(name) for name in BUILDERS}
print("Anchor solves finished for:", ", ".join(anchors))

### 2.1 Visualizing anchor points


In [ ]:
ANCHOR_LABELS = {"f1": "A1", "f2": "A2"}
ANCHOR_GOAL = {"f1": "minimise f1 alone", "f2": "minimise f2 alone"}


def show_table(df, caption):
    """Render a DataFrame as a titled table (falls back to plain text)."""
    try:
        display(df.style.format(precision=4).set_caption(caption))
    except Exception:
        print(f"\n{caption}")
        print(df.round(4).to_string())


def anchor_table(name):
    """One row per anchor: the objective values and the x values behind them."""
    Phi, f_id = anchors[name]
    rows = []
    for j, obj_key in enumerate(["f1", "f2"]):
        row = {"Anchor": ANCHOR_LABELS[obj_key],
               "Objective minimised": ANCHOR_GOAL[obj_key],
               "f1": Phi[0, j],
               "f2": Phi[1, j]}
        row.update(anchor_x[name][obj_key])  # x values at this anchor
        rows.append(row)
    return pd.DataFrame(rows).set_index("Anchor")


for name in BUILDERS:
    display(Markdown(f"#### {name}"))
    show_table(anchor_table(name),
               "Anchor solutions (objective values and decision variables)")


### 2.2 Normalising the objectives

The objective functions are scaled to $[0,1]$ using the ideal and nadir points:

$$
\bar f_i(\mathbf{x}) = \frac{f_i(\mathbf{x}) - f^{id}_i}{f^{nad}_i - f^{id}_i},
\qquad i = 1, 2
$$

Since $f^{id}_i \le f_i \le f^{nad}_i$ holds along the whole front, this maps

- the ideal point to the origin $(0,0)$,
- anchor $A_1$ to $(0,1)$ and anchor $A_2$ to $(1,0)$,
- the nadir point to $(1,1)$,

for **every** problem. The rest of the notebook works in the normalised space, making the four methods and the three problems directly comparable. The map is
affine and increasing, so it changes no solution's Pareto optimality and no front's
shape, only the units on the axes. Original values are recovered at any time from
$f_i = f^{id}_i + \bar f_i\,(f^{nad}_i - f^{id}_i)$.


Rescaling by the ideal and nadir points is standard practice [2,3]. With two
objectives, the nadir point is immediate choice
[4].


In [ ]:
# ======================== OBJECTIVE NORMALISATION ========================

def objective_range(name):
    """(ideal, nadir) for problem `name`, the range each objective covers."""
    Phi, f_id = anchors[name]
    f_nad = np.array([Phi[0, 1], Phi[1, 0]])
    return f_id.copy(), f_nad


SCALING = {name: objective_range(name) for name in BUILDERS}


def normalise(name, values):
    """Map raw objective vectors onto [0, 1]^2:  (f - f_id) / (f_nad - f_id)."""
    values = np.asarray(values, dtype=float).reshape(-1, 2)
    f_id, f_nad = SCALING[name]
    return (values - f_id) / (f_nad - f_id)


def denormalise(name, values):
    """The inverse map, to read a normalised point back in the original units."""
    values = np.asarray(values, dtype=float).reshape(-1, 2)
    f_id, f_nad = SCALING[name]
    return f_id + values * (f_nad - f_id)


def add_normalised_objectives(name, m, comp):
    """Add the two normalised objectives to a GAMSpy model and return them.

    Every method below calls this instead of using comp["f1"] / comp["f2"] directly,
    so each scalarization is built on objectives that share one scale.
    """
    f_id, f_nad = SCALING[name]
    normalised = []
    for k, key in enumerate(("f1", "f2")):
        f_bar = gp.Variable(m, f"{key}_bar", type="free")
        eq = gp.Equation(m, f"eq_{key}_bar")
        # Keep the GAMSpy symbol on the LEFT and feed it plain Python floats: a
        # NumPy scalar on the left of a GAMSpy symbol raises.
        eq[...] = f_bar == (comp[key] - float(f_id[k])) / float(f_nad[k] - f_id[k])
        normalised.append(f_bar)
    return normalised


ANCHORS_BAR = {name: normalise(name, anchors[name][0].T).T for name in BUILDERS}
IDEAL_BAR = np.zeros(2)

scaling_table = pd.DataFrame(
    [{"Problem": name,
      "ideal f1": SCALING[name][0][0], "ideal f2": SCALING[name][0][1],
      "nadir f1": SCALING[name][1][0], "nadir f2": SCALING[name][1][1],
      "f1 range": SCALING[name][1][0] - SCALING[name][0][0],
      "f2 range": SCALING[name][1][1] - SCALING[name][0][1]}
     for name in BUILDERS]).set_index("Problem")
show_table(scaling_table, "Objective ranges used to normalise each problem")

### 2.3 Plotting helper


In [ ]:
# ==================== SHARED PLOTTING / REPORTING HELPERS ====================
# One style for every figure in the notebook. Sizes are set here rather than per
# call, so all panels share the same fonts, and a light grid keeps them readable.
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.titlesize": 14,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.framealpha": 0.9,
})

PANEL_WIDTH, PANEL_HEIGHT = 5.8, 4.8  # one panel of a per-problem figure
LABEL_SIZE = 9                        # point labels and bar labels

# One marker, colour map and colour per method, used by every figure below so that
# a method always looks the same. The colour maps are perceptually uniform and
# colour-blind friendly. Grey is reserved for the reference front and crimson for
# the anchors, so NSGA-II is purple.
METHOD_STYLE = {
    "Weighted sum":       dict(marker="o", cmap="viridis", color="tab:blue"),
    "Epsilon-constraint": dict(marker="s", cmap="plasma",  color="tab:orange"),
    "Modified NBI":       dict(marker="^", cmap="cividis", color="tab:green"),
    "NSGA-II":            dict(marker="D", cmap="viridis", color="tab:purple"),
}

# The reference front is drawn the same way in every panel of every section.
REFERENCE_STYLE = dict(color="0.6", s=6, alpha=0.8)

# The dense reference front is built in section 2.4. If the cache file already
# exists it is borrowed here as a faint background for the per-method plots.
REFERENCE_CACHE = Path("pareto_reference_fronts_normalised.npz")

# Download the precomputed reference fronts once per runtime.
# If the download fails, section 2.4 retains its original computation fallback.
from urllib.request import urlopen
from urllib.error import URLError
from zipfile import BadZipFile
import shutil

REFERENCE_URL = (
    "https://raw.githubusercontent.com/MolChemML/SargentMOO-Tutorial/"
    "main/pareto_reference_fronts_normalised.npz"
)
if not REFERENCE_CACHE.exists():
    temporary_cache = REFERENCE_CACHE.with_suffix(".npz.part")
    try:
        with urlopen(REFERENCE_URL, timeout=30) as response:
            with temporary_cache.open("wb") as target:
                shutil.copyfileobj(response, target)
        # Validate the archive before replacing the cache file.
        with np.load(temporary_cache, allow_pickle=False) as cached:
            for name in BUILDERS:
                front = cached[name]
                if front.ndim != 2 or front.shape[1] != 2 or not np.isfinite(front).all():
                    raise ValueError(f"Invalid reference front: {name}")
                for suffix in ("__f_id", "__f_nadir"):
                    values = cached[name + suffix]
                    if values.shape != (2,) or not np.isfinite(values).all():
                        raise ValueError(f"Invalid scaling data: {name + suffix}")
        temporary_cache.replace(REFERENCE_CACHE)
        print("Downloaded precomputed reference fronts.")
    except (OSError, URLError, ValueError, KeyError, BadZipFile, EOFError) as error:
        print(f"Reference data download failed: {error}")
        print("Section 2.4 will compute the reference fronts; this may take a long time.")
    finally:
        temporary_cache.unlink(missing_ok=True)

_reference_loaded = {}


def get_reference_front(name):
    """Return the cached dense front for `name`, or None if the cache is missing."""
    if not _reference_loaded and REFERENCE_CACHE.exists():
        with np.load(REFERENCE_CACHE) as cached:
            _reference_loaded.update({key: cached[key] for key in cached.files})
    return _reference_loaded.get(name)


def draw_reference_front(ax, name):
    """Draw the best-known front of `name` as the grey background of one panel.

    Every figure calls this, so the reference front never changes appearance.
    """
    front = get_reference_front(name)
    if front is None or not len(front):
        return
    ax.scatter(front[:, 0], front[:, 1], zorder=1,
               label=f"reference front ({len(front)} points)", **REFERENCE_STYLE)


def get_points(method, name):
    """Solutions of `method` on problem `name`, as an (n, 2) array."""
    return np.asarray(results[method].get(name, []), dtype=float).reshape(-1, 2)


def compress_runs(numbers):
    """[1, 2, 3, 5] -> "1-3, 5" so a long list of run numbers stays short."""
    parts = []
    start = previous = numbers[0]
    for number in numbers[1:]:
        if number == previous + 1:
            previous = number
            continue
        parts.append(str(start) if start == previous else f"{start}-{previous}")
        start = previous = number
    parts.append(str(start) if start == previous else f"{start}-{previous}")
    return ", ".join(parts)


def group_labels(points, tol_frac=0.02):
    """Merge markers that sit on top of each other into a single label.

    Several parameter values often return the *same* solution (this is exactly
    what happens to the weighted sum on a non-convex front). Labelling each one
    separately would print an unreadable pile of digits, so coincident points
    share one label such as "1-5".
    """
    span = np.ptp(points, axis=0)
    span = np.where(span > 0, span, 1.0)  # a single point has zero span
    groups = {}
    for run, point in enumerate(points, start=1):
        key = tuple(np.round(point / (tol_frac * span)).astype(int))
        groups.setdefault(key, []).append(run)
    labelled = []
    for members in groups.values():
        centre = points[np.asarray(members) - 1].mean(axis=0)
        labelled.append((centre[0], centre[1], compress_runs(members)))
    return labelled


def plot_method(method, param_label=None, number_points=True):
    """One panel per problem showing the solutions found by `method`."""
    names = list(BUILDERS)
    style = METHOD_STYLE[method]
    fig, axes = plt.subplots(1, len(names),
                             figsize=(PANEL_WIDTH * len(names), PANEL_HEIGHT))
    axes = np.atleast_1d(axes)

    for ax, name in zip(axes, names):
        # Background: the grey reference front built in section 2.4.
        draw_reference_front(ax, name)

        points = get_points(method, name)
        params = np.asarray(method_params[method].get(name, []), dtype=float).ravel()

        if len(points) == 0:
            ax.text(0.5, 0.5, "no solutions returned", transform=ax.transAxes,
                    ha="center", va="center", color="firebrick")
        elif len(params) == len(points):
            # Colour every marker by the parameter value that produced it.
            scatter = ax.scatter(points[:, 0], points[:, 1], c=params, cmap=style["cmap"],
                                 marker=style["marker"], s=90, edgecolor="black",
                                 linewidth=0.6, zorder=3, label=f"{method} solutions")
            bar = fig.colorbar(scatter, ax=ax)
            bar.set_label(param_label or "input parameter")
        else:
            ax.scatter(points[:, 0], points[:, 1], marker=style["marker"], s=55,
                       color=style["color"], edgecolor="black", linewidth=0.4,
                       alpha=0.85, zorder=3, label=f"{method} solutions")

        # The normalised anchors (0, 1) and (1, 0) and the ideal point at the
        # origin give every panel of every problem the same frame of reference.
        Phi_bar = ANCHORS_BAR[name]
        # Hollow stars, so a solution that lands exactly on an anchor still shows.
        ax.scatter(Phi_bar[0], Phi_bar[1], marker="*", s=340, facecolors="none",
                   edgecolors="crimson", linewidth=1.6, zorder=4, label="anchor points")
        ax.scatter([IDEAL_BAR[0]], [IDEAL_BAR[1]], marker="P", s=130, color="darkgreen",
                   edgecolor="black", linewidth=0.5, zorder=4, label="ideal point")

        # Number the points in parameter order when there are few enough to read.
        # A label like "1-5" means runs 1 to 5 all returned the same solution.
        if number_points and 0 < len(points) <= 15:
            for label_x, label_y, text in group_labels(points):
                ax.annotate(text, (label_x, label_y), textcoords="offset points",
                            xytext=(9, 7), fontsize=LABEL_SIZE, color="0.15",
                            bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                                      edgecolor="none", alpha=0.75))

        ax.set_xlabel(r"$\bar f_1$")
        ax.set_ylabel(r"$\bar f_2$")
        ax.set_title(f"{name} ({len(points)} solutions)")
        ax.legend(loc="best")

    fig.suptitle(f"{method}: solutions found in normalised objective space")
    fig.tight_layout()
    plt.show()


def show_solutions(method, param_name=None, max_rows=15):
    """Print the solutions of `method` as one small table per problem."""
    for name in BUILDERS:
        points = get_points(method, name)
        params = np.asarray(method_params[method].get(name, []), dtype=float).ravel()
        if len(points) == 0:
            display(Markdown(f"**{name}**: no solutions returned."))
            continue
        table = pd.DataFrame(points, columns=["f1_bar", "f2_bar"],
                             index=range(1, len(points) + 1))
        if param_name and len(params) == len(points):
            table.insert(0, param_name, params)
        table.index.name = "#"
        if len(table) > max_rows:
            display(Markdown(f"**{name}**: {len(table)} solutions "
                             f"(showing the first {max_rows})"))
            show_table(table.head(max_rows), f"{method} on {name}")
        else:
            show_table(table, f"{method} on {name}")

### 2.4 Best-known reference front (built once, shown on every plot below)

Every plot from here on compares a method's solutions against the same **best-known
Pareto front**, a dense, independent sweep built once now so it is available for
*every* panel, including Method 1's, not only the comparison in section 8. It is
built with its own private $\varepsilon$-constraint sweep (the same idea section 4
teaches, just run at high resolution ahead of time) over `N_REFERENCE_SOLVES`
normalised $\varepsilon$ values, filtered down to its nondominated subset.

The cache is invalidated automatically if it was built from different anchor points
than the ones just computed in section 2 (for example, after a solver change makes
`get_anchor_points` find a better global optimum), not just when the file is
missing, so a stale front from a previous, less accurate run can never be shown
silently.

In [ ]:
# Built once here (rather than only in section 7) so it is available as the grey
# background in every plot below, starting with Method 1's in section 3.
REBUILD_REFERENCE = False
N_REFERENCE_SOLVES = 200        #<------------- can try higher number of solves for denser reference front


def nondominated_unique(points, tol=1e-7):
    points = np.asarray(points, dtype=float).reshape(-1, 2)
    points = np.unique(np.round(points, 8), axis=0)
    points = points[np.argsort(points[:, 0])]
    keep, best_f2 = [], np.inf
    for point in points:
        if point[1] < best_f2 - tol:
            keep.append(point)
            best_f2 = point[1]
    return np.asarray(keep)


def _reference_epsilon_sweep(name, eps_values):
    # A private, minimal epsilon-constraint solve used only to build the dense
    # reference front. Deliberately independent of section 4's own
    # `epsilon_constraint_front` (defined later) so the reference is ready before
    # section 3 needs it, and so section 4 keeps its own from-scratch definition.
    pts = []
    for eps in eps_values:
        m = gp.Container()
        comp = BUILDERS[name](m)
        f1_bar, f2_bar = add_normalised_objectives(name, m, comp)
        eps_con = gp.Equation(m, "eps_con")
        eps_con[...] = f2_bar <= float(eps)
        model = gp.Model(m, name="refmodel", equations=m.getEquations(), problem=PROBLEM_TYPES[name],
                          sense="MIN", objective=f1_bar)
        model.solve()
        if solve_succeeded(model):
            pts.append((comp["f1"].records.level.values[0], comp["f2"].records.level.values[0]))
    return normalise(name, pts) if pts else np.empty((0, 2))


def _cache_is_current(path):
    if not path.exists():
        return False
    with np.load(path) as cached:
        for name in BUILDERS:
            if name not in cached.files or f"{name}__f_id" not in cached.files:
                return False
            f_id, f_nad = SCALING[name]
            if not np.allclose(cached[f"{name}__f_id"], f_id, atol=1e-6):
                return False
            if not np.allclose(cached[f"{name}__f_nadir"], f_nad, atol=1e-6):
                return False
    return True


if REBUILD_REFERENCE or not _cache_is_current(REFERENCE_CACHE):
    dense_eps = np.linspace(0.0, 1.0, N_REFERENCE_SOLVES)
    reference_fronts = {}
    for name in BUILDERS:
        dense = _reference_epsilon_sweep(name, dense_eps)
        reference_fronts[name] = nondominated_unique(dense)
        print(f"Built {name}: {len(reference_fronts[name])} reference points")
    cache_payload = {name: reference_fronts[name] for name in BUILDERS}
    cache_payload.update({f"{name}__f_id": SCALING[name][0] for name in BUILDERS})
    cache_payload.update({f"{name}__f_nadir": SCALING[name][1] for name in BUILDERS})
    np.savez_compressed(REFERENCE_CACHE, **cache_payload)
    print(f"Saved {REFERENCE_CACHE}")
else:
    with np.load(REFERENCE_CACHE) as cached:
        reference_fronts = {name: cached[name] for name in BUILDERS}
    print(f"Loaded cached reference front from {REFERENCE_CACHE}")

## 3. Method 1: Weighted Sum (GAMSpy)

$$\min_{\mathbf{x}\in X}\ w_1 \bar f_1(\mathbf{x}) + w_2 \bar f_2(\mathbf{x}), \qquad w_1+w_2=1,\ w_1,w_2\ge0$$

The weights act on the normalised objectives of section 2.2. Even so, evenly spaced
weights need not give evenly spaced points, and non-convex parts of a front cannot be
reached at all [5].

# To Do

In [ ]:
# Edit this to choose your own 10 weight combinations
w1_values = ... #< EDIT HERE: 10 values in [0, 1] for the weighted sum method

In [ ]:
# GAMSPY METHOD 1: solve one weighted objective for every value of w1.
def weighted_sum_front(name, weights1):
    pts, used_w1 = [], []
    for w1 in weights1:
        w1 = float(w1)
        w2 = 1 - w1  # The two weights must add to 1.

        # Build a fresh copy of the selected benchmark.
        m = gp.Container()
        comp = BUILDERS[name](m)
        f1_bar, f2_bar = add_normalised_objectives(name, m, comp)
        # Combine the two normalised objectives into one objective for GAMSpy.
        obj = gp.Variable(m, "obj", type="free")
        eq_obj = gp.Equation(m, "eq_obj")
        eq_obj[...] = obj == w1 * f1_bar + w2 * f2_bar
        model = gp.Model(m, name="wsmodel", equations=m.getEquations(), problem=PROBLEM_TYPES[name],
                          sense="MIN", objective=obj)
        model.solve()

        # Save the two original objective values, not the weighted value.
        if solve_succeeded(model):
            pts.append((comp["f1"].records.level.values[0], comp["f2"].records.level.values[0]))
            used_w1.append(w1)  # Remember which weight produced this point.
        else:
            print(f"[Weighted sum][{name}] w1={w1:.3f} not solved to optimality: {model.status}")
    return normalise(name, pts), np.asarray(used_w1, dtype=float)


for name in BUILDERS:
    points, used_w1 = weighted_sum_front(name, w1_values)
    results["Weighted sum"][name] = points
    method_params["Weighted sum"][name] = used_w1

# Visualise and list what Method 1 actually produced.
plot_method("Weighted sum", param_label="weight $w_1$")
show_solutions("Weighted sum", param_name="w1")

## 4. Method 2: Epsilon-Constraint (GAMSpy)

$$\min_{\mathbf{x}\in X} \bar f_1(\mathbf{x}) \quad \text{s.t.} \quad \bar f_2(\mathbf{x}) \le \bar\varepsilon$$

# To Do

In [ ]:
# Edit this to choose your own 10 normalised epsilon values, from 0 to 1
eps_fracs = ... #< EDIT HERE: 10 values in [0, 1] for the epsilon-constraint method

In [ ]:
# GAMSPY METHOD 2: minimize f1 while limiting f2 by epsilon.
def epsilon_constraint_front(name, eps_values):
    # The epsilon values are already normalised: 0 asks for the best attainable f2,
    # 1 leaves f2 free up to its nadir value.
    pts, used_eps = [], []
    for eps in eps_values:
        eps = float(eps)
        m = gp.Container()
        comp = BUILDERS[name](m)
        f1_bar, f2_bar = add_normalised_objectives(name, m, comp)
        # Add the extra constraint f2_bar <= epsilon.
        eps_con = gp.Equation(m, "eps_con")
        eps_con[...] = ... #< EDIT HERE: add the constraint f2_bar <= eps
        model = ... #< EDIT HERE: build a GAMSpy model that minimises f1_bar subject to the epsilon constraint on f2_bar
        model.solve()
        if solve_succeeded(model):
            pts.append((comp["f1"].records.level.values[0], comp["f2"].records.level.values[0]))
            used_eps.append(eps)  # Remember which epsilon produced this point.
        else:
            print(f"[Epsilon-constraint][{name}] eps={eps:.3f} not solved to optimality: {model.status}")
    return normalise(name, pts), np.asarray(used_eps, dtype=float)


for name in BUILDERS:
    points, used_eps = epsilon_constraint_front(name, eps_fracs)
    results["Epsilon-constraint"][name] = points
    method_params["Epsilon-constraint"][name] = used_eps
plot_method("Epsilon-constraint", param_label=r"$\bar\varepsilon$ (limit on $\bar f_2$)")
show_solutions("Epsilon-constraint", param_name="eps_bar")

## 5. Method 3: Modified Normal Boundary Intersection (GAMSpy)

Classic NBI [6] solves

$$\max_{\mathbf{x}\in X,\ t} t \quad \text{s.t.} \quad \tilde\Phi\boldsymbol\beta + t\bar n = f(\mathbf{x}) - f^{id}$$

which requires the solution to lie **exactly** on the normal line, a subproblem that can
be infeasible or numerically difficult on non-convex or disconnected fronts. Shukla [7]
relaxes the equality to a component-wise inequality:

$$\max_{\mathbf{x}\in X,\ t} t \quad \text{s.t.} \quad \tilde\Phi\boldsymbol\beta + t\bar n \ge f(\mathbf{x}) - f^{id}$$

which is much easier to solve while still producing (weakly) Pareto-optimal points. Here
$\tilde\Phi=\Phi-f^{id}\mathbf{e}^T$ is the payoff matrix translated to the ideal point,
$\bar n = -\tilde\Phi\mathbf{e}$ is the quasi-normal to the convex hull of individual
minima, and $\boldsymbol\beta=(\beta_1,1-\beta_1)$ are convex weights.

# To Do

In [ ]:
# Edit this to choose your own 10 beta values
beta1_values = ... #< EDIT HERE: 10 values in [0, 1] for the modified NBI method

In [ ]:
# GAMSPY METHOD 3: solve the modified-NBI scalar problem for each beta.
def modified_nbi_front(name, beta1_values):
    # In normalised coordinates the anchors are (0, 1) and (1, 0) and the ideal
    # point is the origin, so the translated payoff matrix is the unit simplex and
    # the quasi-normal comes out as n_bar = -(1, 1) for every problem.
    Phi_tilde = ANCHORS_BAR[name]
    n_bar = -Phi_tilde @ np.ones(2)
    pts, used_beta1 = [], []
    for b1 in beta1_values:
        beta = np.array([b1, 1 - b1])
        rhs = Phi_tilde @ beta
        m = gp.Container()
        comp = BUILDERS[name](m)
        f1_bar, f2_bar = add_normalised_objectives(name, m, comp)
        # t measures how far the solution moves along the NBI direction.
        t = gp.Variable(m, "t", type="free")
        c1 = gp.Equation(m, "nbi1")
        c1[...] = f1_bar <= float(rhs[0]) + t * float(n_bar[0])
        c2 = gp.Equation(m, "nbi2")
        c2[...] = f2_bar <= float(rhs[1]) + t * float(n_bar[1])
        model = gp.Model(m, name="nbimodel", equations=m.getEquations(), problem=PROBLEM_TYPES[name],
                          sense="MAX", objective=t)
        model.solve()
        if solve_succeeded(model):
            pts.append((comp["f1"].records.level.values[0], comp["f2"].records.level.values[0]))
            used_beta1.append(float(b1))  # Remember which beta1 produced this point.
        else:
            print(f"[Modified NBI][{name}] beta1={b1:.3f} not solved to optimality: {model.status}")
    return normalise(name, pts), np.asarray(used_beta1, dtype=float)


for name in BUILDERS:
    points, used_beta1 = modified_nbi_front(name, beta1_values)
    results["Modified NBI"][name] = points
    method_params["Modified NBI"][name] = used_beta1
plot_method("Modified NBI", param_label=r"$\beta_1$")
show_solutions("Modified NBI", param_name="beta1")

## 6. Method 4: NSGA-II (pymoo)

NSGA-II [8,9] is used here through its pymoo implementation [10]. Unlike the
mathematical-programming methods above, it evolves a **population** of candidate
solutions and approximates the whole front at once. One generation uses
the following steps:

1. **Evaluate:** calculate both objectives and every constraint for each candidate.
2. **Select parents:** the continuous problems use binary tournament selection. The
   mixed-variable setup uses random parent selection inside `MixedVariableMating`.
3. **Crossover:** simulated binary crossover (**SBX**) combines two parents. Test 16
   uses SBX followed by rounding repair for its integer variables.
4. **Mutation:** polynomial mutation makes small random changes. Integer
   variables are rounded back to valid integers.
5. **Survival:** parents and children are combined. NSGA-II retains `POP_SIZE`
   candidates using nondominated rank and crowding distance. Duplicate mixed-variable
   solutions are removed for Test 16.

# To Do

In [ ]:
# STUDENT INPUT: edit these to explore how population size and generations
# affect the front
POP_SIZE = 60 # < EDIT HERE: population size for NSGA-II
N_GEN = 300 # < EDIT HERE: number of generations for NSGA-II

In [ ]:
class MOP1Problem(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, n_constr=0, xl=np.array([-3, -3]), xu=np.array([3, 3]))

    def _evaluate(self, x, out, *args, **kwargs):
        # pymoo calls this function for every candidate solution x.
        x1, x2 = x
        out["F"] = [1 / (x1**2 + x2**2 + 1), x1**2 + 3 * x2**2 + 4]


class SCH2Problem(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, n_constr=2, xl=np.array([0, 0]), xu=np.array([5, 3]))

    def _evaluate(self, x, out, *args, **kwargs):
        x1, x2 = x
        f1 = (x1 + x2 - 7.5) ** 2 + (x2 - x1 + 3) ** 2 / 4
        f2 = (x1 - 1) ** 2 / 4 + (x2 - 4) ** 2 / 2
        g1 = (x1 - 2) ** 3 / 2 + x2 - 2.5
        g2 = x1 + x2 - 8 * (x2 - x1 + 0.65) ** 2 - 3.85
        out["F"] = [f1, f2]
        out["G"] = [g1, g2]


class Test16Problem(ElementwiseProblem):
    def __init__(self):
        # Tell pymoo which variables are continuous and which are integers.
        variables = {
            "x1": Real(bounds=(0, 1)),
            "x2": Real(bounds=(0, 1)),
            "x3": Integer(bounds=(-3, 3)),
            "x4": Integer(bounds=(-3, 3)),
        }
        super().__init__(vars=variables, n_obj=2, n_ieq_constr=2)

    def _evaluate(self, x, out, *args, **kwargs):
        x1, x2, x3, x4 = x["x1"], x["x2"], x["x3"], x["x4"]
        out["F"] = [x1 + x3, x2 + x4]
        # pymoo uses g(x) <= 0. The first inequality reverses the outside-ball constraint.
        out["G"] = [1 - x1**2 - x2**2, x3**2 + x4**2 - 9]


# Match each name to its pymoo version.
GA_PROBLEMS = {
    "MOP1": MOP1Problem,
    "SCH2": SCH2Problem,
    "Test 16": Test16Problem,
}


# PYMOO METHOD 4: run the NSGA-II genetic algorithm.
def nsga2_front(name, pop_size, n_gen, seed=1):
    problem = GA_PROBLEMS[name]()
    # Test 16 needs pymoo's mixed-variable operators:
    # Real variables: SBX crossover + polynomial mutation (PM).
    # Integer variables: SBX + PM + rounding to an integer.
    if name == "Test 16":
        duplicates = MixedVariableDuplicateElimination()
        algo = NSGA2(pop_size=pop_size, sampling=MixedVariableSampling(),
                     mating=MixedVariableMating(eliminate_duplicates=duplicates),
                     eliminate_duplicates=duplicates)
    else:
        # pymoo defaults: tournament selection, SBX crossover, PM mutation,
        # and survival based on nondominated rank + crowding distance.
        algo = NSGA2(pop_size=pop_size)
    # Evolve the population for n_gen generations.
    res = pymoo_minimize(problem, algo, ("n_gen", n_gen), seed=seed, verbose=False)
    return res.F


for name in GA_PROBLEMS:
    # NSGA-II searches on the raw objectives, being already scale-invariant. Its
    # output is normalised so that it lands in the same space as the other methods.
    results["NSGA-II"][name] = normalise(name, nsga2_front(name, POP_SIZE, N_GEN))
plot_method("NSGA-II", number_points=False)
show_solutions("NSGA-II")

## 7. The best-known reference front

Built once, right after section 2 (see section 2.4), so it could act as the grey
background in *every* plot from Method 1 onward, not only the section 8
comparison. This section just confirms what was already built.

In [ ]:
# Already built in section 2.4. Nothing to solve again here.
for name in BUILDERS:
    print(f"{name}: {len(reference_fronts[name])} reference points "
          f"(cache: {REFERENCE_CACHE})")

## 8. Comparing all four methods

In [ ]:
fig, axes = plt.subplots(1, len(BUILDERS),
                         figsize=(PANEL_WIDTH * len(BUILDERS), PANEL_HEIGHT))
axes = np.atleast_1d(axes)

# The three scalarizations, drawn with the markers and colours of METHOD_STYLE.
SCALARIZATIONS = ["Weighted sum", "Epsilon-constraint", "Modified NBI"]

for ax, name in zip(axes, BUILDERS):
    draw_reference_front(ax, name)
    ga = results["NSGA-II"][name]
    ax.scatter(ga[:, 0], ga[:, 1], s=15, alpha=0.55, zorder=2,
               color=METHOD_STYLE["NSGA-II"]["color"], label="NSGA-II (population)")
    for method in SCALARIZATIONS:
        pts = results[method][name]
        ax.scatter(pts[:, 0], pts[:, 1], s=38, zorder=3,
                   marker=METHOD_STYLE[method]["marker"],
                   color=METHOD_STYLE[method]["color"], label=method)
    ax.set_xlabel(r"$\bar f_1$")
    ax.set_ylabel(r"$\bar f_2$")
    ax.set_title(name)
    ax.legend()

fig.suptitle("All four methods in normalised objective space")
fig.tight_layout()
plt.show()

## 9. Hypervolume: scoring each method with one number

The plots above compare the solutions obtained by the four methods. The **hypervolume**
(HV) turns that comparison into a single number per method [11]. For a set of points $S$
and a reference point $r$, it is the area of the region dominated by $S$ and bounded by
$r$, so it rewards both closeness to the front and spread along it. The indicator is
*Pareto compliant* [12]: if every point of $A$ is dominated by some point of $B$, then
$HV(A) \le HV(B)$.

**Two choices have to be made, and they must be the same for every method:**

1. **The reference point $r$.** HV is only comparable when all methods use the same $r$.
   Section 2.2 already mapped every problem onto $[0,1]^2$, with the ideal point at the
   origin and the nadir point at $(1,1)$, so a single $r=(1+m,\,1+m)$ serves all three
   problems. We take the margin $m=0.10$ so that a point sitting exactly on an anchor
   still encloses some area.
2. **What counts as 100%.** Even in normalised units the *attainable* area differs from
   problem to problem, because it depends on the shape of the front. We therefore also
   report HV as a percentage of the **best-known front**. That is the nondominated pool
   of the cached reference front plus every point any method found.

In [ ]:
# ============================== HYPERVOLUME ==============================

HV_MARGIN = 0.10  # push the reference point this far beyond the nadir point (1, 1)


def reference_point(margin=HV_MARGIN):
    """The reference point, shared by every method and by every problem.

    Section 2.2 put each problem's ideal point at (0, 0) and its nadir at (1, 1),
    so one point just outside (1, 1) bounds all three problems at once.
    """
    return np.full(2, 1.0 + margin)


def hypervolume_2d(points, ref):
    """Exact bi-objective hypervolume: sum the strips of the dominated staircase."""
    pts = np.asarray(points, dtype=float).reshape(-1, 2)
    pts = pts[np.all(pts <= ref, axis=1)]  # points outside the box dominate nothing
    if len(pts) == 0:
        return 0.0
    pts = pts[np.lexsort((pts[:, 1], pts[:, 0]))]  # sort by f1, ties broken by f2
    area, best_f2 = 0.0, ref[1]
    for f1, f2 in pts:
        if f2 < best_f2:  # only a nondominated point opens a new strip
            area += (ref[0] - f1) * (best_f2 - f2)
            best_f2 = f2
    return area


def best_known_front(name):
    """Nondominated pool of the cached reference front plus every method's points."""
    pool = [reference_fronts[name]] + [get_points(method, name) for method in results]
    return nondominated_unique(np.vstack([p for p in pool if len(p)]))


HV_REF = {name: reference_point() for name in BUILDERS}
HV_BEST_FRONT = {name: best_known_front(name) for name in BUILDERS}
HV_MAX = {name: hypervolume_2d(HV_BEST_FRONT[name], HV_REF[name]) for name in BUILDERS}

# HV of every (method, problem) pair, all against the same reference point.
hv_values = {method: {name: hypervolume_2d(get_points(method, name), HV_REF[name])
                      for name in BUILDERS}
             for method in results}

frame = pd.DataFrame(
    [{"Problem": name,
      "ref r1": HV_REF[name][0], "ref r2": HV_REF[name][1],
      "area of the box": float(np.prod(HV_REF[name])),
      "best-known front": len(HV_BEST_FRONT[name]),
      "HV of best-known front": HV_MAX[name],
      "% of the box": 100 * HV_MAX[name] / float(np.prod(HV_REF[name]))}
     for name in BUILDERS]).set_index("Problem")
show_table(frame, f"Hypervolume reference box, shared by all three problems "
                  f"(margin = {HV_MARGIN:.0%})")

### 9.1 Hypervolume next to the solutions of each problem

In [ ]:
def hv_summary(name):
    """One row per method: how many points it returned and how much area they dominate."""
    rows = []
    for method in results:
        area = hv_values[method][name]
        rows.append({"Method": method,
                     "Solutions": len(get_points(method, name)),
                     "Hypervolume": area,
                     "% of best-known": 100 * area / HV_MAX[name] if HV_MAX[name] else np.nan})
    return pd.DataFrame(rows).set_index("Method")


for name in BUILDERS:
    display(Markdown(f"#### {name}"))
    show_table(hv_summary(name), f"Hypervolume by method (reference point "
                                 f"r = [{HV_REF[name][0]:.2f}, {HV_REF[name][1]:.2f}])")

### 9.2 How hypervolume changes with the problem and the method

In [ ]:
# One panel per problem, as in section 8, with colours taken from METHOD_STYLE.
HV_COLOR = {method: METHOD_STYLE[method]["color"] for method in results}
HV_SHORT = {"Weighted sum": "Weighted\nsum", "Epsilon-constraint": "Epsilon-\nconstraint",
            "Modified NBI": "Modified\nNBI", "NSGA-II": "NSGA-II"}

methods = list(results)
# A shared y-axis across panels is only legitimate because the objectives were
# normalised: the three problems now live in the same reference box.
y_top = max(HV_MAX.values()) * 1.25

fig, axes = plt.subplots(1, len(BUILDERS),
                         figsize=(PANEL_WIDTH * len(BUILDERS), PANEL_HEIGHT))
axes = np.atleast_1d(axes)

for ax, name in zip(axes, BUILDERS):
    areas = [hv_values[method][name] for method in methods]
    counts = [len(get_points(method, name)) for method in methods]
    bars = ax.bar(range(len(methods)), areas,
                  color=[HV_COLOR[method] for method in methods],
                  edgecolor="black", linewidth=0.6, width=0.65, zorder=3)

    # The best front anything in this notebook found = the practical ceiling.
    ax.axhline(HV_MAX[name], color="black", linestyle="--", linewidth=1.2, zorder=4,
               label=f"best-known front = {HV_MAX[name]:.4g}")

    # Print the area, and how much of the ceiling it reaches, on top of each bar.
    for bar, area in zip(bars, areas):
        share = 100 * area / HV_MAX[name] if HV_MAX[name] else 0
        ax.annotate(f"{area:.4g}\n{share:.1f}%",
                    (bar.get_x() + bar.get_width() / 2, area),
                    textcoords="offset points", xytext=(0, 4),
                    ha="center", va="bottom", fontsize=LABEL_SIZE, zorder=6,
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                              edgecolor="none", alpha=0.75))

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels([f"{HV_SHORT[m]}\n(n={c})" for m, c in zip(methods, counts)])
    ax.set_ylabel("Hypervolume (normalised area dominated)")
    ax.set_ylim(0, y_top)
    ax.set_title(name)
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left")

fig.suptitle(f"Hypervolume by method, reference point r = "
             f"({1 + HV_MARGIN:.2f}, {1 + HV_MARGIN:.2f}) for every problem")
fig.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(1.6 * PANEL_WIDTH, PANEL_HEIGHT))
problem_names = list(BUILDERS)
positions = np.arange(len(problem_names))
width = 0.8 / len(methods)

for k, method in enumerate(methods):
    shares = [100 * hv_values[method][name] / HV_MAX[name] if HV_MAX[name] else 0
              for name in problem_names]
    offset = (k - (len(methods) - 1) / 2) * width
    bars = ax.bar(positions + offset, shares, width=width, color=HV_COLOR[method],
                  edgecolor="black", linewidth=0.5, label=method, zorder=3)
    ax.bar_label(bars, fmt="%.1f", fontsize=LABEL_SIZE, padding=3, zorder=6,
                 bbox=dict(boxstyle="round,pad=0.12", facecolor="white",
                           edgecolor="none", alpha=0.75))

ax.axhline(100, color="black", linestyle="--", linewidth=1.2, alpha=0.8, zorder=2,
           label="best-known front")
ax.set_xticks(positions)
ax.set_xticklabels(problem_names)
ax.set_ylabel("Hypervolume as % of the best-known front")
ax.set_ylim(0, 115)
ax.set_title("How hypervolume changes with the problem and with the method")
ax.grid(axis="x", visible=False)
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.07), frameon=False)
fig.tight_layout()
plt.show()

## 10. Discussion questions

1. On **MOP1** (convex front), how similar are the weighted-sum, ε-constraint, and
   modified-NBI fronts? Does spacing between points differ between methods, even
   with evenly spaced input parameters?
2. On **SCH2** (non-convex front), what happens to the weighted-sum points? Why does
   this method fail to explore the middle of the front, even though the objectives are
   normalised?
3. Increase `POP_SIZE` and `N_GEN` for NSGA-II on SCH2. Does the population converge
   more tightly onto the front found by ε-constraint / modified NBI?
4. For **Test 16**, identify which translated quarter-circle arc each computed point
   belongs to. Why can weighted sums miss parts of this disconnected front?
5. Compare the hypervolume table with the plots in sections 3-6. On **SCH2** the
   weighted sum reaches only a fraction of the best-known hypervolume. Which missing
   part of the front costs it the most area?
6. NSGA-II returns 60 points against 10 for the scalarization methods, so it starts
   with an advantage. Recompute the hypervolume of a 10-point subset of the NSGA-II
   population (for example every sixth point). Does it still win?
7. Raise `HV_MARGIN` from 0.10 to 0.5 and re-run. The ranking of the methods should
   barely move, but the percentages change. Why is the *choice of reference point*
   something you must report alongside a hypervolume value?


## 11. References

[1] Eichfelder, Gabriele, Tobias Gerlach, and Leo Warnow. "Test instances for multiobjective mixed-integer nonlinear optimization." In Geometry and Non-Convex Optimization, pp. 71-99. Cham: Springer Nature Switzerland, 2025.

[2] Miettinen, Kaisa. Nonlinear multiobjective optimization. Vol. 12. Boston: Kluwer Academic Publishers, 1999.

[3] Marler, R. Timothy, and Jasbir S. Arora. "The weighted sum method for multi-objective optimization: new insights." Structural and multidisciplinary optimization 41, no. 6 (2010): 853-862.

[4] Ehrgott, Matthias, and Dagmar Tenfelde-Podehl. "Computation of ideal and nadir values and implications for their use in MCDM methods." European Journal of Operational Research 151, no. 1 (2003): 119-139.

[5] Das, Indraneel, and John E. Dennis. "A closer look at drawbacks of minimizing weighted sums of objectives for Pareto set generation in multicriteria optimization problems." Structural optimization 14, no. 1 (1997): 63-69.

[6] Das, Indraneel, and John E. Dennis. "Normal-boundary intersection: A new method for generating the Pareto surface in nonlinear multicriteria optimization problems." SIAM journal on optimization 8, no. 3 (1998): 631-657.

[7] Shukla, Pradyumn Kumar. "On the normal boundary intersection method for generation of efficient front." In International Conference on Computational Science, pp. 310-317. Berlin, Heidelberg: Springer Berlin Heidelberg, 2007.

[8] Deb, Kalyanmoy, Amrit Pratap, Sameer Agarwal, and T. A. M. T. Meyarivan. "A fast and elitist multiobjective genetic algorithm: NSGA-II." IEEE transactions on evolutionary computation 6, no. 2 (2002): 182-197.

[9] Deb, K. "Multi-objective optimization using evolutionary algorithms. Chichester: John Wiley and Sons, Inc." (2001).

[10] Blank, Julian, and Kalyanmoy Deb. "Pymoo: Multi-objective optimization in python." Ieee access 8 (2020): 89497-89509.

[11] Zitzler, Eckart, and Lothar Thiele. "Multiobjective evolutionary algorithms: a comparative case study and the strength Pareto approach." IEEE transactions on Evolutionary Computation 3, no. 4 (1999): 257-271.

[12] Zitzler, Eckart, Lothar Thiele, Marco Laumanns, Carlos M. Fonseca, and Viviane Grunert Da Fonseca. "Performance assessment of multiobjective optimizers: An analysis and review." IEEE Transactions on evolutionary computation 7, no. 2 (2003): 117-132.